# BlocoRocha A.02: Momentum Growth, equal weights, monthly rebalancing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tiago-baptistalopes/bloco-rocha/blob/main/notebooks/A02_equal_weight_monthly.ipynb)

| Item | Rule |
|---|---|
| Universe | Every dollar-listed United States and euro-listed European firm with a market value of at least €1.5 billion at fund creation, 2005-01-25 (set `N_PER_REGION` to sample instead) |
| Entry | Top quartile 12-1 month momentum **and** top half price-to-book; filled to 50 names by combined rank |
| Exit | Debt-to-equity up 100% or more year on year, or price-to-book up by 10% or less year on year; firms showing either signal cannot enter |
| Rebalancing | Monthly (on the 25th, or the next trading day): entry, exit checks and reset to equal weights across the 50 holdings (`REBALANCE_MONTHS = 12` for annual) |
| Listings | United States stocks listed in dollars and European stocks listed in euros only |
| Currency | Euro; United States dollar positions fully hedged (local returns, carry ignored) |
| Risk-free rate | 12-month EURIBOR (European Central Bank monthly average), used in Sharpe, Sortino and Omega ratios and shown as a cash line |
| Benchmarks | S&P 500 total return (dollar, treated as hedged like the fund's dollar positions); equal-weighted universe |
| Persona tests | Maximum drawdown no worse than −35%; value-at-risk-equivalent volatility 12–20% (Summary Risk Indicator class 4) |

Prices: yfinance. Fundamentals: Compustat via Wharton Research Data Services (cached to `data/fundamentals.csv`).

In [57]:
%pip install -q yfinance quantstats wrds

Note: you may need to restart the kernel to use updated packages.


In [58]:
import os
import time
import logging
import numpy as np
import pandas as pd
import yfinance as yf
import quantstats as qs
import matplotlib.pyplot as plt

START = pd.Timestamp('2005-01-25')
HISTORY_START = '2003-12-01'
N_HOLD = 50
N_PER_REGION = None
MIN_MCAP_EUR = 1.5e9
MOM_Q, PB_Q = 0.75, 0.50
DE_JUMP, PB_DROP = 1.0, 0.10
MDD_LIMIT, VEV_RANGE, RHP_YEARS = -0.35, (0.12, 0.20), 20
SEED = 42
CURRENCIES = ('USD', 'EUR')
WEIGHTING = 'equal'
REBALANCE_MONTHS = 1
EURO_SUFFIXES = {'AS', 'BR', 'PA', 'DE', 'F', 'HM', 'MU', 'BE', 'DU', 'SG', 'MI', 'MC', 'LS', 'IR', 'VI', 'HE', 'AT', 'LU'}
SUFFIX_CCY = {'L': 'GBp', 'SW': 'CHF', 'ST': 'SEK', 'CO': 'DKK', 'OL': 'NOK', 'WA': 'PLN'}

def currency(t):
    if '.' not in t:
        return 'USD'
    suffix = t.rsplit('.', 1)[1]
    return 'EUR' if suffix in EURO_SUFFIXES else SUFFIX_CCY.get(suffix, suffix)
WRDS_USERNAME = ''
DATA = 'data'
os.makedirs(DATA, exist_ok=True)

## 1. Universe: every firm above the size threshold at fund creation

All United States and European firms in Compustat with fiscal year 2004 data, pre-screened at a market value of €1 billion, then screened exactly at €1.5 billion on 2005-01-25 with yfinance prices. Compustat includes firms delisted since 2005; the share that yfinance cannot price measures the survivorship bias.

In [59]:
def cached(name, fetch, dates=True):
    path = f'{DATA}/{name}.csv'
    if os.path.exists(path):
        return pd.read_csv(path, index_col=0, parse_dates=dates)
    df = fetch()
    if df.empty or df.isna().all().any():
        print(f'{name}: download empty or incomplete, not cached; rerun later')
    else:
        df.to_csv(path)
    return df

CCYS = ['USD', 'GBP', 'CHF', 'SEK', 'DKK', 'NOK', 'PLN']

def fetch_fx():
    fx = yf.download([f'EUR{c}=X' for c in CCYS], start=HISTORY_START, auto_adjust=True, progress=False)['Close']
    fx.columns = [c[3:6] for c in fx.columns]
    return fx

fx = cached('fx', fetch_fx).ffill().bfill()
fx['EUR'] = 1.0

In [60]:
EUROPE = ['GBR','FRA','DEU','NLD','BEL','LUX','IRL','ESP','PRT','ITA','CHE','AUT','SWE','DNK','NOR','FIN','GRC']
FUNDA_US = "f.indfmt='INDL' and f.datafmt='STD' and f.popsrc='D' and f.consol='C'"
FUNDA_EU = "f.indfmt='INDL' and f.datafmt='HIST_STD' and f.popsrc='I' and f.consol='C'"

def wrds_query(sql):
    import wrds
    db = wrds.Connection(wrds_username=WRDS_USERNAME or None)
    if not os.path.exists(os.path.expanduser('~/.pgpass')):
        db.create_pgpass_file()
    df = db.raw_sql(sql)
    db.close()
    return df

def fetch_compustat_universe():
    return wrds_query(f'''
        select f.gvkey, f.tic as symbol, 'US' as region, f.curcd, f.prcc_f * f.csho * 1e6 as mcap_local
        from comp.funda f join comp.company c on c.gvkey = f.gvkey
        where {FUNDA_US} and f.fyear = 2004 and c.loc = 'USA' and f.prcc_f * f.csho >= 1000
        union all
        select f.gvkey, s.isin as symbol, 'EU' as region, d.curcdd as curcd, d.prccd * d.cshoc as mcap_local
        from comp.g_funda f
        join comp.g_company c on c.gvkey = f.gvkey
        join comp.g_security s on s.gvkey = f.gvkey and s.iid = c.prirow
        join lateral (select prccd, cshoc, curcdd from comp.g_secd x
                      where x.gvkey = s.gvkey and x.iid = s.iid and x.datadate <= f.datadate and x.datadate > f.datadate - 10
                      order by x.datadate desc limit 1) d on true
        where {FUNDA_EU} and f.fyear = 2004 and c.loc in ({','.join(f"'{x}'" for x in EUROPE)})''').set_index('gvkey')

cu = cached('compustat_universe', fetch_compustat_universe, dates=False)
cu.index = cu.index.astype(str).str.zfill(6)
cu = cu[~cu.index.duplicated(keep='last')]
rate = fx.loc[:'2004-12-31'].iloc[-1]
cu['mcap_eur'] = cu['mcap_local'] / cu['curcd'].map(rate)
cu = cu[cu['mcap_eur'] >= 1e9]
cu['region'].value_counts()

region
US    1288
EU     548
Name: count, dtype: int64

In [61]:
def yahoo_symbol(isin):
    try:
        quotes = yf.Search(isin, max_results=5).quotes
        return quotes[0]['symbol'] if quotes else 'NOT_FOUND'
    except Exception:
        return None

path = f'{DATA}/yahoo_map.csv'
ymap = pd.read_csv(path, index_col=0, dtype=str)['ticker'] if os.path.exists(path) else pd.Series(dtype=str, name='ticker')
ymap.index = ymap.index.astype(str).str.zfill(6)
todo = [g for g in cu.index if g not in ymap.index or pd.isna(ymap.get(g))]
for g in todo:
    s_, r = cu.loc[g, 'symbol'], cu.loc[g, 'region']
    ymap[g] = s_.replace('.', '-') if r == 'US' else yahoo_symbol(s_)
    if r == 'EU':
        time.sleep(0.5)
if todo:
    ymap.rename('ticker').to_csv(path)
print(f'{ymap.isna().sum()} lookups failed (rerun this cell to retry), {(ymap == "NOT_FOUND").sum()} not on Yahoo Finance')
ymap = ymap[ymap.notna() & (ymap != 'NOT_FOUND')]
ymap = ymap[ymap.index.isin(cu.index) & ~ymap.duplicated()]
found = cu.loc[ymap.index, 'region'].value_counts()
listing_ok = [(currency(t) == 'USD') if cu.loc[g, 'region'] == 'US' else (currency(t) == 'EUR') for g, t in ymap.items()]
ymap = ymap[[ok and currency(t) in CURRENCIES for ok, t in zip(listing_ok, ymap)]]
TICKERS = sorted(ymap)
REGION = pd.Series(cu.loc[ymap.index, 'region'].values, index=ymap.values)
GVKEY = pd.Series(ymap.index, index=ymap.values)

0 lookups failed (rerun this cell to retry), 201 not on Yahoo Finance


## 2. Prices and currencies (yfinance)

In [62]:
class LogCapture(logging.Handler):
    def __init__(self):
        super().__init__()
        self.lines = []
    def emit(self, record):
        self.lines.append(record.getMessage())

yf_log = LogCapture()
logging.getLogger('yfinance').addHandler(yf_log)

def download(tickers, batch=100, pause=60, retries=3):
    out, limited = [], []
    for i in range(0, len(tickers), batch):
        chunk = tickers[i:i + batch]
        for attempt in range(retries):
            yf_log.lines.clear()
            close = yf.download(chunk, start=HISTORY_START, auto_adjust=True, progress=False, threads=True)['Close']
            close = (close.to_frame(chunk[0]) if isinstance(close, pd.Series) else close).reindex(columns=chunk)
            hit = [t for t in chunk if any('Rate' in l and f"'{t}'" in l for l in yf_log.lines)]
            if not hit:
                break
            time.sleep(pause)
        limited += hit
        out.append(close.drop(columns=hit))
        time.sleep(2)
    if limited:
        print(f'{len(limited)} tickers still rate-limited; rerun this cell later to fetch only those')
    return pd.concat(out, axis=1)

path = f'{DATA}/prices.csv'
px = pd.read_csv(path, index_col=0, parse_dates=True) if os.path.exists(path) else pd.DataFrame()
missing = [t for t in TICKERS if t not in px.columns]
if missing:
    px = pd.concat([px, download(missing)], axis=1)
    px.to_csv(path)
empty = [t for t in TICKERS if t not in px.columns or px[t].isna().all()]
print(f'{len(TICKERS)} tickers, {len(TICKERS) - len(empty)} with prices, {len(empty)} without (delisted or rate-limited)')

1493 tickers, 870 with prices, 623 without (delisted or rate-limited)


In [63]:
px = px[[t for t in TICKERS if t in px.columns and px[t].notna().any()]].dropna(how='all')
CCY = pd.Series({t: currency(t) for t in px.columns})
fx = fx.reindex(px.index).ffill().bfill()

def to_eur(t):
    c = CCY[t]
    return px[t] * (0.01 if c == 'GBp' else 1.0) / fx['GBP' if c == 'GBp' else c]

px_eur = pd.DataFrame({t: to_eur(t) for t in px.columns})
px_ret = pd.DataFrame({t: px[t] if CCY[t] == 'USD' else px_eur[t] for t in px.columns})
CCY.value_counts()

USD    665
EUR    205
Name: count, dtype: int64

## 3. Fundamentals (Compustat via Wharton Research Data Services)

Debt = long-term debt + debt in current liabilities. Fiscal year $y-1$ is used at the January rebalance of year $y$ (accepted look-ahead bias). Compustat amounts are in millions.

In [64]:
def fetch_fundamentals():
    keys = ','.join(f"'{g}'" for g in cu.index)
    return wrds_query(f'''
        select f.gvkey, f.fyear, f.curcd, f.ceq * 1e6 as ceq, (coalesce(f.dltt,0) + coalesce(f.dlc,0)) * 1e6 as debt,
               f.csho * 1e6 as shares
        from comp.funda f where {FUNDA_US} and f.fyear between 2002 and 2026 and f.gvkey in ({keys})
        union all
        select f.gvkey, f.fyear, f.curcd, f.ceq * 1e6, (coalesce(f.dltt,0) + coalesce(f.dlc,0)) * 1e6, d.cshoc
        from comp.g_funda f
        join comp.g_company c on c.gvkey = f.gvkey
        left join lateral (select cshoc from comp.g_secd x
                           where x.gvkey = f.gvkey and x.iid = c.prirow and x.datadate <= f.datadate and x.datadate > f.datadate - 10
                           order by x.datadate desc limit 1) d on true
        where {FUNDA_EU} and f.fyear between 2002 and 2026 and f.gvkey in ({keys})''').set_index('gvkey')

fund = cached('fundamentals', fetch_fundamentals, dates=False).reset_index()
fund['gvkey'] = fund['gvkey'].astype(str).str.zfill(6)
tick = pd.Series(GVKEY.index, index=GVKEY.values)
fund['ticker'] = fund['gvkey'].map(tick)
fund = fund.dropna(subset=['ticker', 'ceq']).drop_duplicates(['ticker', 'fyear'], keep='last')
ceq, debt, shares = (fund.pivot(index='fyear', columns='ticker', values=v) for v in ('ceq', 'debt', 'shares'))
curcd = fund.groupby('ticker')['curcd'].last()
de = (debt / ceq).where(ceq > 0)

## 4. Exact size screen at fund creation and annual signals

In [65]:
pxf = px_eur.ffill(limit=10)
pxr = px_ret.ffill(limit=10)
DATES = sorted({px.index[px.index >= m][0] for m in pd.date_range(START, px.index[-1], freq=pd.DateOffset(months=REBALANCE_MONTHS))})

def at(df, d):
    rows = df.loc[:d]
    return rows.iloc[-1] if len(rows) else pd.Series(np.nan, index=df.columns)

def book_eur(d):
    y = d.year - 1
    return ceq.reindex([y]).iloc[0] / curcd.reindex(ceq.columns).map(at(fx, d)).astype(float)

def mcap_eur(d):
    return at(pxf, d).reindex(shares.columns) * shares.reindex([d.year - 1]).iloc[0]

def pb(d):
    b = book_eur(d)
    return (mcap_eur(d) / b).where(b > 0)

def momentum(d):
    return at(pxr, d - pd.DateOffset(months=1)) / at(pxr, d - pd.DateOffset(months=12)) - 1

mcap0 = mcap_eur(DATES[0]).dropna()
UNIVERSE = sorted(mcap0[mcap0 >= MIN_MCAP_EUR].index)
if N_PER_REGION:
    rng = np.random.default_rng(SEED)
    UNIVERSE = [t for r in ('US', 'EU') for t in rng.choice(sorted(t for t in UNIVERSE if REGION[t] == r),
                size=min(N_PER_REGION, sum(REGION[t] == r for t in UNIVERSE)), replace=False)]

funnel = pd.DataFrame({
    'Compustat, pre-screen above €1bn': cu['region'].value_counts(),
    'Found on Yahoo Finance': found,
    'Dollar-listed (United States) or euro-listed (Europe)': REGION.value_counts(),
    'Priced on 2005-01-25': REGION[at(pxf, DATES[0]).dropna().index.intersection(REGION.index)].value_counts(),
    'Universe (above €1.5bn)': REGION[UNIVERSE].value_counts()}).T
funnel['Total'] = funnel.sum(axis=1)
funnel

,US,EU,Total
"Compustat, pre-screen above €1bn",1288,548,1836
Found on Yahoo Finance,1288,346,1634
Dollar-listed (United States) or euro-listed (Europe),1288,205,1493
Priced on 2005-01-25,617,195,812
Universe (above €1.5bn),302,98,400


## 5. Portfolio rules

At each rebalance date $t$ (monthly): holdings that trigger an exit signal are sold,
$$\frac{D/E_{y-1}}{D/E_{y-2}} - 1 \ge 1.0 \quad \text{or} \quad \frac{P/B_t}{P/B_{t-12\text{ months}}} - 1 \le 0.10, $$
the remaining holdings are kept, and free slots are filled first with names in the top momentum quartile and the top price-to-book half, then with the best remaining combined rank $\tfrac{1}{2}\left(\text{rank}_{\text{momentum}} + \text{rank}_{P/B}\right)$. Holdings are then reset to equal weights, $w_i = 1/N$; weights drift with prices until the next monthly rebalance.

In [66]:
holdings, log = {}, []
held = []
for d in DATES:
    alive = at(pxf, d).reindex(UNIVERSE).dropna().index
    mom, pbs = momentum(d).reindex(alive), pb(d).reindex(alive)
    y = d.year - 1
    de_jump = (de.loc[y] / de.loc[y - 1] - 1 >= DE_JUMP).reindex(alive, fill_value=False) if {y, y - 1} <= set(de.index) else pd.Series(False, index=alive)
    pb_drop = (pbs / pb(d - pd.DateOffset(years=1)).reindex(alive) - 1 <= PB_DROP)
    signal = de_jump | pb_drop
    exits = [t for t in held if t not in alive or signal.get(t, False)]
    held = [t for t in held if t not in exits]
    ok = mom.dropna().index.intersection(pbs.dropna().index).difference(signal[signal].index)
    score = (mom[ok].rank(pct=True) + pbs[ok].rank(pct=True)) / 2
    passed = (mom[ok] >= mom[ok].quantile(MOM_Q)) & (pbs[ok] >= pbs[ok].quantile(PB_Q))
    order = pd.DataFrame({'passed': passed, 'score': score}).sort_values(['passed', 'score'], ascending=False).index
    entries = [t for t in order if t not in held][:max(N_HOLD - len(held), 0)]
    held = held + entries
    if WEIGHTING == 'market_cap':
        mc = mcap_eur(d).reindex(held)
        w = mc.fillna(mc.median()) / mc.fillna(mc.median()).sum()
    else:
        w = pd.Series(1 / len(held), index=held)
    holdings[d] = w
    log.append({'date': d, 'holdings': len(held), 'entries': len(entries), 'exits': len(exits),
                'exit_leverage': sum(bool(de_jump.get(t, False)) for t in exits),
                'exit_price_to_book': sum(bool(pb_drop.get(t, False)) for t in exits),
                'passed_both_filters': int(passed.sum()),
                'largest_weight': w.max(), 'effective_holdings': 1 / (w ** 2).sum()})
log = pd.DataFrame(log).set_index('date')
log.groupby(log.index.year).agg({'entries': 'sum', 'exits': 'sum', 'exit_leverage': 'sum', 'exit_price_to_book': 'sum',
                                  'passed_both_filters': 'mean', 'largest_weight': 'max', 'effective_holdings': 'min'}).round(2)

,entries,exits,exit_leverage,exit_price_to_book,passed_both_filters,largest_weight,effective_holdings
date,,,,,,,
2005,91,41,0,41,21.25,0.02,50.0
2006,83,83,5,78,24.00,0.02,50.0
2007,83,83,4,80,19.58,0.02,50.0
2008,104,133,7,129,5.42,0.08,12.0
2009,68,39,3,37,13.83,0.07,15.0
2010,52,52,1,51,29.17,0.02,50.0
2011,83,83,1,82,18.67,0.02,50.0
2012,53,53,5,48,20.75,0.02,50.0
2013,34,34,3,31,30.08,0.02,50.0


## 6. Backtest

In [ ]:
def backtest(weights, prices):
    prices = prices.ffill()  # carry the last close over market holidays, so a holding that does not trade on a rebalance day (e.g. European stocks on 26 December, US stocks on Memorial Day) is not dropped for the month
    dates = list(weights) + [prices.index[-1]]
    parts = []
    for d0, d1 in zip(dates[:-1], dates[1:]):
        w = weights[d0]
        p = prices.loc[d0:d1, w.index].ffill()
        parts.append((p / p.iloc[0]).mul(w, axis=1).sum(axis=1).pct_change().dropna())
    return pd.concat(parts)

fund_r = backtest(holdings, px_ret).rename('BlocoRocha A.02')
bench_holdings = {}
for d in DATES:
    alive = at(pxf, d).reindex(UNIVERSE).dropna().index
    bench_holdings[d] = pd.Series(1 / len(alive), index=alive)
bench_r = backtest(bench_holdings, px_ret).rename('Universe equal weight')

def fetch_sp500():
    return yf.download('^SP500TR', start=HISTORY_START, auto_adjust=True, progress=False)['Close']

sp500 = cached('sp500', fetch_sp500).iloc[:, 0]
sp500_r = sp500.reindex(px.index.union(sp500.index)).ffill().pct_change().reindex(fund_r.index).rename('S&P 500 total return')

cum = (1 + pd.concat([fund_r, sp500_r, bench_r], axis=1)).cumprod()
cum = pd.concat([pd.DataFrame(1.0, index=[DATES[0]], columns=cum.columns), cum])
ax = cum.plot(figsize=(11, 5), logy=True, title='Cumulative returns (start = 1, log scale)')
ax.axhline(1, color='grey', lw=0.8)
plt.show()
cum.iloc[-1]

## Risk-free rate: 12-month EURIBOR

European Central Bank Data Portal series `FM.M.U2.EUR.RT.MM.EURIBOR1YD_.HSTA` (monthly average, percent per annum), applied to every trading day of its month. quantstats converts it to a daily rate, $r_{f,d} = (1 + r_{f})^{1/252} - 1$, for the excess-return ratios; cumulative returns stay gross.

In [ ]:
ECB_URL = 'https://data-api.ecb.europa.eu/service/data/FM/M.U2.EUR.RT.MM.EURIBOR1YD_.HSTA?format=csvdata&startPeriod=2003-12'

def fetch_euribor():
    raw = pd.read_csv(ECB_URL)
    return raw.set_index(pd.to_datetime(raw['TIME_PERIOD']))[['OBS_VALUE']].rename(columns={'OBS_VALUE': 'euribor_12m'})

euribor = cached('euribor_12m', fetch_euribor)['euribor_12m'] / 100
rf = euribor.reindex(euribor.index.union(fund_r.index)).ffill().reindex(fund_r.index)
cash_r = ((1 + rf) ** (1 / 252) - 1).rename('12-month EURIBOR (cash)')

cum = (1 + pd.concat([fund_r, sp500_r, bench_r, cash_r], axis=1)).cumprod()
cum = pd.concat([pd.DataFrame(1.0, index=[DATES[0]], columns=cum.columns), cum])
ax = cum.plot(figsize=(11, 5), logy=True, title='Cumulative returns (start = 1, log scale)')
ax.axhline(1, color='grey', lw=0.8)
plt.show()
pd.DataFrame({'Average 12-month EURIBOR': [rf.mean()], 'Minimum': [rf.min()], 'Maximum': [rf.max()]}).round(4)

In [ ]:
strategies = pd.concat([fund_r, bench_r], axis=1)
m = qs.reports.metrics(strategies, benchmark=sp500_r, rf=rf, mode='full', display=False)
m.loc['Treynor Ratio'] = qs.reports.metrics(strategies, benchmark=sp500_r, rf=float(rf.mean()), mode='full', display=False).loc['Treynor Ratio']
m

## 7. Persona tests

Value-at-risk-equivalent volatility (Commission Delegated Regulation (EU) 2017/653, Annex II), with $\sigma$, $\mu_1$ (skewness) and $\mu_2$ (excess kurtosis) of daily log returns, $N = 256\,T$ trading days over the recommended holding period $T$:
$$\text{VaR}_{\text{return}} = \sigma\sqrt{N}\left(-1.96 + 0.474\frac{\mu_1}{\sqrt{N}} - 0.0687\frac{\mu_2}{N} + 0.146\frac{\mu_1^2}{N}\right) - 0.5\,\sigma^2 N, \qquad \text{VEV} = \frac{\sqrt{3.842 - 2\,\text{VaR}_{\text{return}}} - 1.96}{\sqrt{T}}$$

In [ ]:
def vev(r, T=RHP_YEARS):
    x = np.log1p(r.dropna())
    s, m1, m2, N = x.std(), x.skew(), x.kurt(), 256 * T
    var = s * np.sqrt(N) * (-1.96 + 0.474 * m1 / np.sqrt(N) - 0.0687 * m2 / N + 0.146 * m1**2 / N) - 0.5 * s**2 * N
    return (np.sqrt(3.842 - 2 * var) - 1.96) / np.sqrt(T)

def persona_tests(r):
    mdd, v = qs.stats.max_drawdown(r), vev(r)
    return pd.DataFrame({
        'value': [mdd, v, qs.stats.volatility(r), qs.stats.value_at_risk(r)],
        'target': [f'>= {MDD_LIMIT:.0%}', f'{VEV_RANGE[0]:.0%} to {VEV_RANGE[1]:.0%}', 'reference', 'reference (daily, 95%)'],
        'pass': [mdd >= MDD_LIMIT, VEV_RANGE[0] <= v <= VEV_RANGE[1], None, None]},
        index=['Maximum drawdown', 'Value-at-risk-equivalent volatility', 'Annualised volatility', 'Historical value at risk'])

tests = persona_tests(fund_r)
print('Persona conditions met:', bool(tests['pass'].dropna().all()))
pd.concat({'BlocoRocha A.02': tests, 'S&P 500 total return': persona_tests(sp500_r), 'Universe Equal Weights': persona_tests(bench_r)}, axis=1)

## 8. Backtest analysis

Monthly holdings, trading costs and the performance measures required by the brief. Assumptions used in every table and figure of this section:

| Item | Assumption |
|---|---|
| Return frequency | Monthly returns, compounded from daily returns within each calendar month (the first and last months are partial). Daily fund and S&P 500 returns are not synchronous (European and United States markets close at different times and have different holidays), which biases daily beta and correlation downwards; set `FREQ = 'D'` to check robustness |
| Annualisation | Arithmetic mean × 12 and volatility × √12; the compound (geometric) return over the whole period is also shown; beta is not annualised |
| Risk-free rate | 12-month EURIBOR (European Central Bank), converted to a daily rate and compounded within each month like the portfolio |
| Sharpe ratio | Annualised mean excess return over the risk-free rate / annualised volatility of excess returns |
| Alpha and beta | Ordinary least squares regression of monthly fund excess returns on benchmark excess returns; alpha × 12, with its t-statistic |
| Information ratio | Annualised mean active return (fund − benchmark) / annualised tracking error (volatility of active returns × √12) |
| Transaction costs | `COST_BPS` per euro traded plus `FEE_PER_ORDER` per order, charged at each rebalance including the initial purchase. Trades = target weight − weight drifted with prices since the previous rebalance |
| Benchmarks | Primary: S&P 500 total return. Secondary: equal-weighted universe |

The quantstats table in Section 6 uses other definitions (for example, its information ratio is not annualised and its alpha is not estimated on excess returns), so the prospectus reports the tables below.

In [ ]:
# 8.0 Data check: largest daily price move of each universe stock (moves above ±50% are usually bad yfinance data)
p = pxr[UNIVERSE]
moves = (p / p.shift(1) - 1).fillna(0.0)
worst = moves.abs().idxmax()
check = pd.DataFrame({'largest daily move': [moves.loc[d, t] for t, d in worst.items()],
                      'date': worst.values, 'region': REGION.reindex(worst.index).values}, index=worst.index)
print(f'{(moves.abs() > 0.5).any().sum()} of {len(UNIVERSE)} universe stocks have at least one daily move above ±50%')
check.loc[check['largest daily move'].abs().sort_values(ascending=False).index].head(15)

In [ ]:
# 8.1 Monthly holdings: target weights set at each rebalance and the weights just before it
def drifted_weights(weights, prices):
    """Weight of each holding at the next rebalance date, before trading, after drifting with prices."""
    prices = prices.ffill()  # carry the last close over market holidays, as in backtest()
    dates = list(weights)
    pre = {dates[0]: pd.Series(dtype=float)}  # nothing is held before the first purchase
    for d0, d1 in zip(dates[:-1], dates[1:]):
        w = weights[d0]
        p = prices.loc[d0:d1, w.index]
        v = w * p.iloc[-1] / p.iloc[0]
        pre[d1] = v / v.sum()
    return pd.DataFrame(pre).T

W = pd.DataFrame(holdings).T.fillna(0.0)  # rows: rebalance dates, columns: tickers
W_pre = drifted_weights(holdings, px_ret).reindex(index=W.index, columns=W.columns).fillna(0.0)
trades = W - W_pre                        # + bought, − sold, as a share of portfolio value
traded = trades.abs().sum(axis=1)         # buys + sells (first rebalance: initial purchase = 100%)
orders = (trades.abs() > 1e-6).sum(axis=1)

rows = []
for d in W.index:
    names = W.columns[(W.loc[d] > 0) | (W_pre.loc[d] > 0)]
    rows.append(pd.DataFrame({'date': d, 'ticker': names, 'region': REGION.reindex(names).values,
                              'weight before': W_pre.loc[d, names].values, 'target weight': W.loc[d, names].values}))
holdings_long = pd.concat(rows, ignore_index=True)  # full month-by-month list (also usable by the app)
holdings_long['action'] = np.select([holdings_long['weight before'] == 0, holdings_long['target weight'] == 0],
                                    ['enter', 'exit'], 'keep')

us = REGION.reindex(W.columns).eq('US').values
holdings_summary = pd.DataFrame({'holdings': (W > 0).sum(axis=1), 'United States weight': W.loc[:, us].sum(axis=1),
                                 'Europe weight': W.loc[:, ~us].sum(axis=1), 'traded (buys + sells)': traded,
                                 'orders': orders})
print(f'{len(W)} rebalances; holdings at the latest one ({W.index[-1]:%Y-%m-%d}):')
holdings_long[holdings_long['date'] == W.index[-1]].sort_values(['action', 'ticker'])

In [ ]:
# Monthly summary (holdings count, regional split, trading); the full list is in `holdings_long`
holdings_summary.round(3)

In [ ]:
# 8.2 Transaction costs. PLACEHOLDERS: the group must set and justify these values (decisions log: still open)
COST_BPS = 10        # proportional cost per euro traded, in basis points (bid-ask spread and commission)
FEE_PER_ORDER = 0.0  # fixed broker fee per order, in euros
CAPITAL = 3_000      # client's initial investment in euros (decisions log: €2,500–€3,000)

def net_returns(r, traded, orders, bps=COST_BPS, fee=FEE_PER_ORDER, capital=CAPITAL):
    """Daily returns after costs charged at each rebalance: bps × value traded + fee × number of orders."""
    day = [r.index[r.index >= d][0] for d in traded.index]  # rebalance close (initial purchase: first return day)
    prop = pd.Series(traded.values * bps / 1e4, index=day).groupby(level=0).sum().reindex(r.index, fill_value=0.0)
    n = pd.Series(orders.values, index=day).groupby(level=0).sum().reindex(r.index, fill_value=0)
    out, v = [], float(capital)  # v: portfolio value in euros
    for x, c, k in zip(r.to_numpy(), prop.to_numpy(), n.to_numpy()):
        if v <= 0:                  # fixed fees have used up the whole portfolio
            out.append(0.0)
            continue
        v0 = v
        v = max(v * (1 + x) * (1 - c) - k * fee, 0.0)
        out.append(v / v0 - 1)
    return pd.Series(out, index=r.index)

fund_net = net_returns(fund_r, traded, orders).rename('BlocoRocha A.02 (net of costs)')
years = (fund_r.index[-1] - DATES[0]).days / 365.25
drag = (1 + fund_r).prod() ** (1 / years) - (1 + fund_net).prod() ** (1 / years)
print(f'Average one-way turnover: {traded.iloc[1:].sum() / 2 / years:.0%} of portfolio value per year')
print(f'Average orders per rebalance: {orders.iloc[1:].mean():.0f}')
print(f'Cost drag on the compound return: {drag:.2%} per year ({COST_BPS} bps per euro traded, €{FEE_PER_ORDER} per order, €{CAPITAL:,} invested)')
pd.DataFrame({'one-way turnover': traded.groupby(traded.index.year).sum() / 2,
              'orders': orders.groupby(orders.index.year).sum()}).round(2)

In [ ]:
# 8.3 Performance measures required by the brief (Tables 8.3a and 8.3b); assumptions at the top of Section 8
import statsmodels.api as sm

FREQ = 'M'  # 'M' monthly (reported) or 'D' daily (robustness check)
PERIODS = {'D': 252, 'M': 12}
PCT = ['Annualised average return (arithmetic)', 'Annualised compound return (geometric)', 'Annualised volatility',
       'Alpha (annualised)', 'Tracking error (annualised)', 'Maximum drawdown (daily)']
RELATIVE = ['Alpha (annualised)', 'Alpha t-statistic', 'Beta (not annualised)', 'Tracking error (annualised)', 'Information ratio']

def performance(r, bench, rf=cash_r, freq=FREQ):
    """Brief's measures for one daily return series against one benchmark (rf: daily risk-free return)."""
    k = PERIODS[freq]
    daily = pd.concat({'p': r, 'b': bench, 'rf': rf}, axis=1).dropna()
    df = daily if freq == 'D' else (1 + daily).resample('ME').prod() - 1
    ex, ex_b, active = df['p'] - df['rf'], df['b'] - df['rf'], df['p'] - df['b']
    same = active.abs().max() < 1e-12  # the benchmark measured against itself
    fit = None if same else sm.OLS(ex, sm.add_constant(ex_b)).fit()
    te = active.std() * np.sqrt(k)
    wealth = (1 + daily['p']).cumprod()
    years = (daily.index[-1] - daily.index[0]).days / 365.25
    return pd.Series({
        'Annualised average return (arithmetic)': df['p'].mean() * k,
        'Annualised compound return (geometric)': wealth.iloc[-1] ** (1 / years) - 1,
        'Annualised volatility': df['p'].std() * np.sqrt(k),
        'Sharpe ratio': ex.mean() / ex.std() * np.sqrt(k),
        'Alpha (annualised)': 0.0 if same else fit.params.iloc[0] * k,
        'Alpha t-statistic': np.nan if same else fit.tvalues.iloc[0],
        'Beta (not annualised)': 1.0 if same else fit.params.iloc[1],
        'Tracking error (annualised)': te,
        'Information ratio': np.nan if same else active.mean() * k / te,
        'Maximum drawdown (daily)': (wealth / wealth.cummax().clip(lower=1) - 1).min(),
    })

def fmt(t):
    return pd.DataFrame([['–' if pd.isna(x) else f'{x:.2%}' if i in PCT else f'{x:.2f}' for x in t.loc[i]] for i in t.index],
                        index=t.index, columns=t.columns)

series = {'A.02 gross': fund_r, 'A.02 net of costs': fund_net,
          'S&P 500 total return': sp500_r, 'Universe equal weight': bench_r}
table_sp = pd.DataFrame({n: performance(s, sp500_r) for n, s in series.items()})
table_ew = pd.DataFrame({n: performance(series[n], bench_r) for n in ['A.02 gross', 'A.02 net of costs']}).loc[RELATIVE]
freq_label = 'monthly' if FREQ == 'M' else 'daily'
print(f'Table 8.3a: measures against the S&P 500 total return ({freq_label} returns, '
      f'{fund_r.index[0]:%Y-%m-%d} to {fund_r.index[-1]:%Y-%m-%d})')
display(fmt(table_sp))
print('Table 8.3b: relative measures against the equal-weighted universe')
display(fmt(table_ew))

In [ ]:
# 8.4 Figure: cumulative performance (start = 1) and drawdowns against the client's limit
from matplotlib.ticker import FuncFormatter, NullFormatter, PercentFormatter

STYLE = {'A.02 net of costs': ('#2a78d6', '-'), 'A.02 gross': ('#2a78d6', '--'),
         'S&P 500 total return': ('#eb6834', '-'), 'Universe equal weight': ('#1baf7a', '-')}
lines = pd.concat(series, axis=1).dropna()
wealth = pd.concat([pd.DataFrame(1.0, index=[DATES[0]], columns=lines.columns), (1 + lines).cumprod()])
drawdown = wealth / wealth.cummax() - 1

fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
for name, (color, ls) in STYLE.items():
    a1.plot(wealth.index, wealth[name], color=color, ls=ls, lw=1.6 if ls == '-' else 1.1,
            label=f'{name} (final value {wealth[name].iloc[-1]:.2f})')
    if name != 'A.02 gross':
        a2.plot(drawdown.index, drawdown[name], color=color, lw=1.2)
a1.set_yscale('log')
a1.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f'{y:g}'))
a1.yaxis.set_minor_formatter(NullFormatter())
a1.axhline(1, color='#9b9a97', lw=0.8)
a1.set_title('Cumulative performance (start = 1, log scale)', loc='left')
a1.legend(frameon=False, loc='upper left')
a2.axhline(MDD_LIMIT, color='#52514e', ls=':', lw=1)
a2.annotate(f"Client's drawdown limit ({MDD_LIMIT:.0%})", (drawdown.index[-1], MDD_LIMIT), xytext=(0, -4),
            textcoords='offset points', ha='right', va='top', fontsize=8, color='#52514e')
a2.set_title('Drawdown from previous peak', loc='left')
a2.yaxis.set_major_formatter(PercentFormatter(1))
for a in (a1, a2):
    a.grid(axis='y', color='#e6e5e1', lw=0.6)
    a.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# 8.5 Calendar-year returns (Table 8.5); 2005 and 2026 are partial years
yearly = (1 + lines).groupby(lines.index.year).prod() - 1
yearly['A.02 net − S&P 500'] = yearly['A.02 net of costs'] - yearly['S&P 500 total return']
print(f"Net of costs, A.02 beat the S&P 500 in {(yearly['A.02 net − S&P 500'] > 0).sum()} of {len(yearly)} calendar years")
yearly.map(lambda x: f'{x:.1%}')

In [ ]:
# 8.6 Cost sensitivity (Table 8.6): illustrative cost levels, all against the S&P 500 total return
grid = []
for bps in (0, 10, 25, 50):
    for fee in (0, 1, 3):
        m = performance(net_returns(fund_r, traded, orders, bps=bps, fee=fee), sp500_r)
        grid.append({'cost per euro traded (bps)': bps, 'fee per order (€)': fee,
                     'compound return': m['Annualised compound return (geometric)'], 'Sharpe ratio': m['Sharpe ratio'],
                     'alpha': m['Alpha (annualised)'], 'information ratio': m['Information ratio']})
print(f'Fixed fees are a share of a €{CAPITAL:,} portfolio, so their effect shrinks as the portfolio grows')
pd.DataFrame(grid).set_index(['cost per euro traded (bps)', 'fee per order (€)']).style.format(
    {'compound return': '{:.2%}', 'alpha': '{:.2%}', 'Sharpe ratio': '{:.2f}', 'information ratio': '{:.2f}'})